Run All Cells to reproduce the Rosenbrock Benchmark Figure from "Direct From Darwin". Runtime: 5 minutes on Cloud CPU

In [ ]:
import sys
if 'google.colab' in sys.modules:
    !git clone https://github.com/danielgrimmer/adam-dls.git
    sys.path.insert(0, '/content/adam-dls')


In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

from adam_dls import AdamDLS
from baselines.noisy_adam import NoisyAdam
import torch
import numpy as np
import matplotlib.pyplot as plt

# =====================================================================
# Generating Five Gradient Ascent Trajectories on Rosenbrock
# =====================================================================

def rosenbrock(tensor):
    x, y = tensor[0], tensor[1]
    a = 2
    return ((a - x)**2 + 100 * (y - x**2)**2)

# Tracking dictionaries
trajectories = {'SGD1': [],'SGD2': [],'SGD3': [],'SGD4': [],'SGD5': []}
losses = {'SGD1': [],'SGD2': [],'SGD3': [],'SGD4': [],'SGD5': []}
extra_histories = {}

# 1. Noisy Gradient Ascent (Five Times)
sgd_steps = 2000
sgd_lr = 3e-6      # ||H||_2 = 3402 at (-2,4) forces lr <~ 0.01/||H||_2 = 3e-6
sgd_mu2 = 1e-4     # Mutation Rate ultimately set by Adam-DLS to ensure W_g >= 0 (see Diagnostic Panel)

start_coords = [-1.5, 2.5]
t_sgd1 = torch.tensor(start_coords, requires_grad=True)
opt_sgd1 = NoisyAdam([t_sgd1], lr=sgd_lr, betas=(0,0), mu_sq = sgd_mu2, just_SGD=True, minimize=True)
print("Optimizing Gradient Ascent 1...")
for i in range(sgd_steps):
    if i % 2000 == 0:
        print(f"SGD1: {i}/{sgd_steps}")

    opt_sgd1.zero_grad()
    loss = rosenbrock(t_sgd1)
    loss.backward()
    opt_sgd1.step()

    trajectories['SGD1'].append(t_sgd1.detach().clone().numpy())
    losses['SGD1'].append(loss.item())

start_coords = [-0.9, 1.0]
t_sgd2 = torch.tensor(start_coords, requires_grad=True)
opt_sgd2 = NoisyAdam([t_sgd2], lr=sgd_lr, betas=(0,0), mu_sq = sgd_mu2, just_SGD=True, minimize=True)
print("Optimizing Gradient Ascent 2...")
for i in range(sgd_steps):
    if i % 2000 == 0:
        print(f"SGD2: {i}/{sgd_steps}")

    opt_sgd2.zero_grad()
    loss = rosenbrock(t_sgd2)
    loss.backward()
    opt_sgd2.step()

    trajectories['SGD2'].append(t_sgd2.detach().clone().numpy())
    losses['SGD2'].append(loss.item())

start_coords = [0.0, 0.1]
t_sgd3 = torch.tensor(start_coords, requires_grad=True)
opt_sgd3 = NoisyAdam([t_sgd3], lr=sgd_lr, betas=(0,0), mu_sq = sgd_mu2, just_SGD=True, minimize=True)
print("Optimizing Gradient Ascent 3...")
for i in range(sgd_steps):
    if i % 2000 == 0:
        print(f"SGD3: {i}/{sgd_steps}")

    opt_sgd3.zero_grad()
    loss = rosenbrock(t_sgd3)
    loss.backward()
    opt_sgd3.step()

    trajectories['SGD3'].append(t_sgd3.detach().clone().numpy())
    losses['SGD3'].append(loss.item())

start_coords = [0.9, 1.0]
t_sgd4 = torch.tensor(start_coords, requires_grad=True)
opt_sgd4 = NoisyAdam([t_sgd4], lr=sgd_lr, betas=(0,0), mu_sq = sgd_mu2, just_SGD=True, minimize=True)
print("Optimizing Gradient Ascent 4...")
for i in range(sgd_steps):
    if i % 2000 == 0:
        print(f"SGD4: {i}/{sgd_steps}")

    opt_sgd4.zero_grad()
    loss = rosenbrock(t_sgd4)
    loss.backward()
    opt_sgd4.step()

    trajectories['SGD4'].append(t_sgd4.detach().clone().numpy())
    losses['SGD4'].append(loss.item())

start_coords = [1.5, 2.5]
t_sgd5 = torch.tensor(start_coords, requires_grad=True)
opt_sgd5 = NoisyAdam([t_sgd5], lr=sgd_lr, betas=(0,0), mu_sq = sgd_mu2, just_SGD=True, minimize=True)
print("Optimizing Gradient Ascent 5...")
for i in range(sgd_steps):
    if i % 2000 == 0:
        print(f"SGD5: {i}/{sgd_steps}")

    opt_sgd5.zero_grad()
    loss = rosenbrock(t_sgd5)
    loss.backward()
    opt_sgd5.step()

    trajectories['SGD5'].append(t_sgd5.detach().clone().numpy())
    losses['SGD5'].append(loss.item())


In [ ]:
# =====================================================================
# Generating Adam and Adam-DLS Trajectories on Rosenbrock
# =====================================================================


steps = 500000              # Maximum steps for Adam and Adam-DLS
start_coords = [-1.9, 4.1]
alpha = 1e-3                # Learning Rate ultimately set by Adam-DLS to ensure Tr(V_g H_g) <~ 0.01 (see Diagnostic Panel)
betas = (0.9, 0.999)
mu2_adams = 1e-4            # Mutation Rate ultimately set by Adam-DLS to ensure W_g >= 0 (see Diagnostic Panel)

# Initialize Adam
trajectories['Adam'] = []
losses['Adam'] = []
t_adam = torch.tensor(start_coords, requires_grad=True)
opt_adam = NoisyAdam([t_adam], lr=alpha, betas=betas, mu_sq=mu2_adams, just_SGD=False, minimize=True)

print("Optimizing Adam...")
adam_steps = steps
for i in range(adam_steps):

    opt_adam.zero_grad()
    loss = rosenbrock(t_adam)
    loss.backward()
    opt_adam.step()

    trajectories['Adam'].append(t_adam.detach().clone().numpy())
    losses['Adam'].append(loss.item())

    if i % 10000 == 0:
        print(f"Adam: {i}/{adam_steps}  Loss: ",f"{loss.item():.2f}")

    #Simple Convergence Criteria
    if loss.item() < 2e-3:
        break

print('Done with Adam!')

# Initialize Adam-DLS
trajectories['Adam-DLS'] = []
losses['Adam-DLS'] = []
t_adamdls = torch.tensor(start_coords, requires_grad=True)
opt_adamdls = AdamDLS([t_adamdls], lr=alpha, betas=betas, mu_sq=mu2_adams, record_history=True, minimize=True)

print("\n Optimizing Adam-DLS...")
adamdls_steps = steps
for i in range(adamdls_steps):

    opt_adamdls.zero_grad()
    loss = rosenbrock(t_adamdls)
    loss.backward()
    opt_adamdls.step()

    trajectories['Adam-DLS'].append(t_adamdls.detach().clone().numpy())
    losses['Adam-DLS'].append(loss.item())

    if i % 10000 == 0:
        print(f"Adam-DLS: {i}/{adamdls_steps} Loss: ",f"{loss.item():.2f}")

    #Simple Convergence Criteria
    if loss.item() < 2e-3:
        break

#Records produced by Adam-DLS for Diagnostics Panel
extra_histories['AdamDLS_D_g_history'] = opt_adamdls.D_g_history
extra_histories['AdamDLS_m_g_history'] = opt_adamdls.m_g_history
extra_histories['AdamDLS_d_history'] = opt_adamdls.d_history
extra_histories['AdamDLS_mu_history'] = opt_adamdls.mu_history

print('Done with Adam-DLS!')

In [ ]:
# ======================================================================
# Generating Rosenbrock Benchmark Figure from "Direct From Darwin" Paper
# ======================================================================

import matplotlib.patches as patches
from matplotlib.patches import Circle
from matplotlib.patches import Ellipse

colors = {
    'SGD1': '#FF0000',          # Red
    'SGD2': '#FF7F00',          # Orange
    'SGD3': '#FFFF00',          # Yellow
    'SGD4': '#00FF00',          # Green
    'SGD5': '#0000FF',          # Blue
    'Adam': '#0000cc',          # Dark Blue
    'Adam-DLS': '#cc0000'       # Dark Red
}

fig = plt.figure(figsize=(18, 12)) # Adjust figure size for 3x2 panels

# Common settings for all trajectory plots
X = np.linspace(-2.5, 2.5, 100)
Y = np.linspace(-0.5, 4.5, 100)
X_mesh, Y_mesh = np.meshgrid(X, Y)

X_tensor = torch.from_numpy(X_mesh).float()
Y_tensor = torch.from_numpy(Y_mesh).float()
Z_tensor = rosenbrock(torch.stack([X_tensor, Y_tensor]))
Z = Z_tensor.numpy()

# Shared parameters for reference objects (circles, arrows, ellipses)
n_objects = 9
object_x_coords = np.linspace(-2, 2, n_objects)
object_y_coord = -0.9

# Panel 1 (Top-Left): Five Gradient Ascent Trajectories
ax1 = plt.subplot(2, 3, 1)
ax1.set_xlim(-2.5, 2.5)
ax1.set_ylim(-0.5, 4.5)
contour_set1 = ax1.contour(X_mesh, Y_mesh, np.log10(Z + 1e-8), 40, cmap='binary_r', alpha=0.5)
ax1.set_xticks([])
ax1.set_yticks([])
ax1.set_aspect('equal', adjustable='box')
ax1.set_rasterized(True)

# Plot all Five Gradient Ascent trajectories
for name in ['SGD1', 'SGD2', 'SGD3', 'SGD4', 'SGD5']:
    if name in trajectories:
        traj_sgd = np.array(trajectories[name])
        ax1.plot(traj_sgd[:, 0], traj_sgd[:, 1], label=name, color=colors[name], alpha=0.8, linewidth=2)
ax1.plot(2.0, 4.0, '*', markersize=12, color='gold', label='Global Optimum')
ax1.set_title("Noisy Gradient Ascent", fontsize=30)

# Panel 2 (Top-Middle): Noisy Adam Trajectory
ax2 = plt.subplot(2, 3, 2)
ax2.set_xlim(-2.5, 2.5)
ax2.set_ylim(-0.5, 4.5)
contour_set2 = ax2.contour(X_mesh, Y_mesh, np.log10(Z + 1e-8), 40, cmap='binary_r', alpha=0.5)
ax2.set_xticks([])
ax2.set_yticks([])
ax2.set_aspect('equal', adjustable='box')
ax2.set_rasterized(True)
if 'Adam' in trajectories:
    traj_adam = np.array(trajectories['Adam'])
    ax2.plot(traj_adam[:, 0], traj_adam[:, 1], label='Adam', color=colors['Adam'], alpha=0.8, linewidth=2)
ax2.plot(2.0, 4.0, '*', markersize=12, color='gold', label='Global Optimum')
ax2.set_title("Noisy Adam Trajectory", fontsize=30)

# Panel 3 (Top-Right): Adam-DLS Trajectory
ax3 = plt.subplot(2, 3, 3)
ax3.set_xlim(-2.5, 2.5)
ax3.set_ylim(-0.5, 4.5)
contour_set3 = ax3.contour(X_mesh, Y_mesh, np.log10(Z + 1e-8), 40, cmap='binary_r', alpha=0.5)
ax3.set_xticks([])
ax3.set_yticks([])
ax3.set_aspect('equal', adjustable='box')
ax3.set_rasterized(True)
if 'Adam-DLS' in trajectories:
    traj_adamdls = np.array(trajectories['Adam-DLS'])
    ax3.plot(traj_adamdls[:, 0], traj_adamdls[:, 1], label='Adam-DLS', color=colors['Adam-DLS'], alpha=0.8, linewidth=2)
ax3.plot(2.0, 4.0, '*', markersize=12, color='gold', label='Global Optimum')
ax3.set_title("Adam-DLS Trajectory", fontsize=30)


# Panel 4 (Bottom-Left): Circles Showing Mutation and Variance Scales for Gradient Ascent
ax4 = plt.subplot(2, 3, 4)
ax4.set_xlim(-2.5, 2.5)
ax4.set_ylim(object_y_coord - 0.35, object_y_coord + 0.35)
ax4.set_yticks([])
ax4.set_xticks([])
ax4.set_aspect('equal', adjustable='box')
ax4.set_xlabel('Mutation and Variance Scales\n (20x Magnification)', fontsize=26)
for spine in ax4.spines.values():
    spine.set_visible(False)

scale = 20                          # Magnification Scale
radii = [scale*0.0017] * n_objects  # With pop. variance 3e-6 the stdev is 0.0017

for i, x_center in enumerate(object_x_coords):
    circle = Circle((x_center, object_y_coord),
                    radius=radii[i],
                    facecolor='gainsboro',
                    edgecolor='dimgray',
                    linewidth=1.5,
                    zorder=3)
    if i == 0:
        ax4.add_patch(Circle((x_center, object_y_coord),
              radius=scale*0.01,    # With mut. variance 1e-4 the stdev is 0.01
              facecolor='magenta',
              edgecolor='purple',
              linewidth=1.5,
              zorder=3))
    ax4.add_patch(circle)

# Panel 5 (Bottom-Middle): Arrows for Adam
ax5 = plt.subplot(2, 3, 5)
ax5.set_xlim(-2.5, 2.5)
ax5.set_ylim(object_y_coord - 0.35, object_y_coord + 0.35)
ax5.set_yticks([])
ax5.set_xticks([])
ax5.set_aspect('equal', adjustable='box')
ax5.set_xlabel('Orientation of Adam\'s\n Momentum: $D_g m_g$', fontsize=26)
for spine in ax5.spines.values():
    spine.set_visible(False)

# Initialize arrays
dx_data = np.zeros(n_objects)
dy_data = np.zeros(n_objects)

# Scale factor to control the visual length of the arrows on the plot
arrow_scale = 0.6

for i, x in enumerate(object_x_coords):
    # Calculate the magnitude of the tangent vector of ridge (1, 2x)
    magnitude = np.sqrt(1 + (2 * x)**2)

    # Normalize and scale
    dx_data[i] = (1 / magnitude) * arrow_scale
    dy_data[i] = ((2 * x) / magnitude) * arrow_scale

for i, x_start in enumerate(object_x_coords):
    dx = dx_data[i]
    dy = dy_data[i]

    arrow = patches.FancyArrowPatch((x_start - dx/2, object_y_coord - dy/2),
                                    (x_start + dx/2, object_y_coord + dy/2),
                                    mutation_scale=15,
                                    color='black',
                                    arrowstyle='-|>',
                                    linewidth=1.5)
    ax5.add_patch(arrow)

# Panel 6 (Bottom-Right): Ellipses for Adam-DLS
ax6 = plt.subplot(2, 3, 6)
ax6.set_xlim(-2.5, 2.5)
ax6.set_ylim(object_y_coord - 0.35, object_y_coord + 0.35)
ax6.set_yticks([])
ax6.set_xticks([])
ax6.set_aspect('equal', adjustable='box')
ax6.set_xlabel('Orientation of Adam-DLS\'s\n Lineage Variance: $V_g$', fontsize=26)
for spine in ax6.spines.values():
    spine.set_visible(False)

# Define the major and minor axes based on the condition number (numerically estimatated at ~10)
ellipse_major_axis = 0.6
ellipse_minor_axis = ellipse_major_axis / 10.0

widths = [ellipse_major_axis] * n_objects
heights = [ellipse_minor_axis] * n_objects
angles = np.degrees(np.arctan2(dy_data, dx_data))

for i, x_center in enumerate(object_x_coords):
    ellipse = Ellipse((x_center, object_y_coord),
                      width=widths[i],
                      height=heights[i],
                      angle=angles[i],
                      facecolor='gainsboro',
                      edgecolor='dimgrey',
                      linewidth=1.5,
                      alpha=0.6,
                      zorder=3)
    ax6.add_patch(ellipse)


plt.tight_layout(h_pad=-20.0)

plt.savefig(
    'AdamDLSRosenbrockBenchmark.pdf',
    format='pdf',
    bbox_inches='tight',
    pad_inches=0.1,
    dpi = 300
    )

plt.show()

In [ ]:
# ======================================================================
# Generating Rosenbrock Benchmark Diagnostics
# ======================================================================

# Note: this diagnostic cell builds full N×N matrices and is only appropriate for small N
def compute_V_g(m_g, D_g, beta1):
    """
    Reconstruct the Adam-DLS lineage variance V_g from recorded
    momentum m_g and preconditioner diagonal D_g.

    V_g = (1 - beta1) * diag(D_g)
        + beta1 * (D_g m_g)(D_g m_g)^T / (m_g^T D_g m_g)

    The second term is omitted (replaced with zero) when m_g^T D_g m_g
    is too small to divide safely, which occurs at initialisation.
    """
    D_g_m_g = D_g * m_g                                    # element-wise, shape (N,)
    m_T_D_m  = torch.sum(m_g * D_g_m_g)                    # scalar

    V_g = (1 - beta1) * torch.diag(D_g)                    # (N, N) diagonal part

    numerator = torch.outer(D_g_m_g, D_g_m_g)              # (N, N) rank-1 part
    rank1     = beta1 * numerator / (m_T_D_m + 1e-15)

    V_g += torch.where(
        m_T_D_m > 1e-12,
        rank1,
        torch.zeros_like(rank1)
    )
    return V_g


def compute_hessian(x, y):
    # Hessian of the Rosenbrock function F(x, y) = (2 - x)^2 + 100 * (y - x**2)**2
    # H_xx = d^2F/dx^2 = 2 + 1200*x**2 - 400*y
    # H_xy = d^2F/dxdy = -400*x
    # H_yx = d^2F/dydx = -400*x
    # H_yy = d^2F/dy^2 = 200

    H_xx = 2 + 1200 * x**2 - 400 * y
    H_xy = -400 * x
    H_yx = -400 * x # Corrected: H_yx was missing
    H_yy = 200

    hessian_matrix = torch.tensor([[H_xx, H_xy],
                                   [H_yx, H_yy]], dtype=torch.float32)
    return hessian_matrix

# Retrieve histories and beta1 from global scope
adamdls_m_history = extra_histories['AdamDLS_m_g_history']
adamdls_D_history = extra_histories['AdamDLS_D_g_history']
adamdls_traj = trajectories['Adam-DLS']
beta1_val = betas[0]

assert len(adamdls_m_history) == len(adamdls_D_history) == len(adamdls_traj), \
    "History lists must have the same length."

trace_product_history = []

for i in range(len(adamdls_m_history)):
    m_g  = adamdls_m_history[i].cpu()
    D_g  = adamdls_D_history[i].cpu()
    x, y = adamdls_traj[i][0], adamdls_traj[i][1]

    V_g = compute_V_g(m_g, D_g, beta1_val)
    H_g = compute_hessian(x, y)

    trace_value = torch.trace(V_g @ H_g)
    trace_product_history.append(trace_value.item())

# Create a new figure for the diagnostics plots (Loss Curves and Mutation Rate)
fig_diagnostics = plt.figure(figsize=(18, 12))

# Panel 1: Loss Curves
ax1_diag = fig_diagnostics.add_subplot(2, 2, 1)
for name in ['Adam-DLS','Adam']:
    ax1_diag.plot(losses[name], label=f'{name}', color=colors[name], linewidth=2)
ax1_diag.set_yscale('log')
ax1_diag.set_title("Loss over Generations",fontsize=20)
ax1_diag.set_xlabel("Generations (g)")
ax1_diag.set_ylabel("Rosenbrock Loss")
ax1_diag.legend()

# Panel 2: Mutation Rate (mu_sq)
ax2_diag = fig_diagnostics.add_subplot(2, 2, 2)

# Plot mu_history for Adam-DLS
if 'AdamDLS_mu_history' in extra_histories:
    mu_raw_dls = extra_histories['AdamDLS_mu_history']
    ax2_diag.plot(mu_raw_dls, color=colors['Adam-DLS'], linewidth=2, label='Threshold for Evolutionary Compliance')

# Add a horizontal line at the actual mutation rate
ax2_diag.axhline(mu2_adams, color='k', linestyle='--', alpha=0.5, label='Actual Mutation Rate')

ax2_diag.set_title("Actual Mutation Rate and Threshold for $W_g \geq 0$",fontsize=20)
ax2_diag.set_xlabel("Generations (g)")
ax2_diag.set_ylabel("Mutation Rate")
ax2_diag.set_yscale('log')
ax2_diag.legend()

# Panel 3: d scalar history for Adam-DLS (Histogram)
ax3_diag = fig_diagnostics.add_subplot(2, 2, 3)

if 'AdamDLS_d_history' in extra_histories:
    d_raw_dls = extra_histories['AdamDLS_d_history']

    # Calculate mean and variance
    d_mean = np.mean(d_raw_dls)
    d_stdev = np.sqrt(np.var(d_raw_dls))

    # Controls for the histogram bins
    d_scalar_hist_min = d_mean - 1 * d_stdev
    d_scalar_hist_max = d_mean + 1 * d_stdev
    d_scalar_hist_bins = 50

    # Plot histogram of d scalar values
    ax3_diag.hist(d_raw_dls, bins=d_scalar_hist_bins, range=(d_scalar_hist_min, d_scalar_hist_max),
                  color=colors['Adam-DLS'], density=True)
    ax3_diag.set_title("Histogram of d Scalar",fontsize=20)
    ax3_diag.set_xlabel("d Scalar Value")
    ax3_diag.set_ylabel("Density")

    # Add vertical dashed line at d=1
    ax3_diag.axvline(x=1, color='gray', linestyle='--', linewidth=2, label='d=1')

    # Add a box with mean and variance
    textstr = f'Mean: {d_mean:.2f}\nStandard Dev.: {d_stdev:.2f}'
    props = dict(boxstyle='round,pad=0.5', facecolor='white', alpha=0.8)
    ax3_diag.text(0.05, 0.95, textstr, transform=ax3_diag.transAxes, fontsize=10,
            verticalalignment='top', bbox=props)
    ax3_diag.legend()

# Panel 4: Trace(V_g @ H_g)
ax4_diag = fig_diagnostics.add_subplot(2, 2, 4)
ax4_diag.plot(trace_product_history, linewidth=2, color=colors['Adam-DLS'])
ax4_diag.axhline(0, color='k', linewidth=0.5, linestyle='--')
ax4_diag.set_title(r"Evolutionary Fidelity Condition: $\mathrm{Tr}(V_g H_g) \lesssim 0.01$", fontsize=20)
ax4_diag.set_xlabel("Generations (g)") # Use "Generations (g)" for consistency
ax4_diag.set_ylabel(r"$\mathrm{Tr}(V_g H_g)$")
textstr = f'Mean Trace: {torch.tensor(trace_product_history).abs().mean():.6f}'
props = dict(boxstyle='round,pad=0.5', facecolor='white', alpha=0.8)
ax4_diag.text(0.05, 0.95, textstr, transform=ax4_diag.transAxes, fontsize=10,
        verticalalignment='top', bbox=props)


plt.tight_layout()
plt.show()